# Origin of Temperature Change

Use compact ERA5 downloads from CDS to explain a 12-hour temperature-change window around Kansas City, Missouri.

The table focuses on hour-to-hour 2 m temperature change. Horizontal advection is computed directly from neighboring ERA5 grid cells as `-(u dT/dx + v dT/dy)`. The other attribution columns allocate the non-advection residual by evidence scores from the small ERA5 variable set, so treat them as diagnostic triage rather than a closed surface energy budget.

In [1]:
from pathlib import Path
import os
import zipfile

import numpy as np
import pandas as pd
import xarray as xr

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)


def find_project_root(start):
    for path in [start, *start.parents]:
        if (path / "AGENTS.md").exists():
            return path
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
repo_tmp = PROJECT_ROOT / "tmp"
data_dir = PROJECT_ROOT / "data"
(repo_tmp / "matplotlib").mkdir(parents=True, exist_ok=True)
(repo_tmp / "era5-extracted").mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(repo_tmp / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(repo_tmp / "cache"))

'/home/dmmsp/Projects/observations-vs-forecasts/tmp/cache'

In [2]:
POINT_NAME = "Kansas City, MO"
POINT_LAT = 39.0997
POINT_LON = -94.5783

ANALYSIS_DATE = "2017-03-14"
ANALYSIS_HOURS = list(range(4, 16))
CENTER_HOUR = 9

SURFACE_PATH = data_dir / "kansas-city-era5-surface-diagnostics_2017-03.nc"
PRESSURE_PATH = data_dir / "kansas-city-era5-850-hpa-diagnostics_2017-03_p850.nc"

print(f"Point: {POINT_NAME} ({POINT_LAT:.4f}, {POINT_LON:.4f})")
print(f"ERA5 analysis window: {ANALYSIS_DATE} {ANALYSIS_HOURS[0]:02d}:00-{ANALYSIS_HOURS[-1]:02d}:00 UTC")
print(f"Surface file: {SURFACE_PATH.name}")
print(f"850 hPa file: {PRESSURE_PATH.name}")

Point: Kansas City, MO (39.0997, -94.5783)
ERA5 analysis window: 2017-03-14 04:00-15:00 UTC
Surface file: kansas-city-era5-surface-diagnostics_2017-03.nc
850 hPa file: kansas-city-era5-850-hpa-diagnostics_2017-03_p850.nc


## ERA5 Variable Set

This notebook uses two compact ERA5 files prepared with the ERA5 request-planner helper:

- Single levels: 2 m temperature, 10 m wind, boundary layer height, total cloud cover, downwelling shortwave/longwave radiation, total precipitation, total column water vapour, snow depth, soil water layer 1, lake cover, land-sea mask, and geopotential.
- Pressure levels: 850 hPa temperature, wind, vertical velocity, and specific humidity.

In [3]:
G = 9.80665
R_EARTH = 6_371_229.0


def open_surface_parts(path):
    extract_dir = repo_tmp / "era5-extracted" / path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)

    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            archive.extractall(extract_dir)
        instant_path = extract_dir / "data_stream-oper_stepType-instant.nc"
        accum_path = extract_dir / "data_stream-oper_stepType-accum.nc"
        return (
            xr.open_dataset(instant_path, engine="netcdf4"),
            xr.open_dataset(accum_path, engine="netcdf4"),
        )

    ds = xr.open_dataset(path, engine="netcdf4")
    return ds, ds


def select_time(ds):
    wanted = pd.to_datetime([f"{ANALYSIS_DATE} {hour:02d}:00" for hour in ANALYSIS_HOURS])
    return ds.sel(valid_time=wanted)


surface_instant, surface_accum = open_surface_parts(SURFACE_PATH)
surface_instant = select_time(surface_instant)
surface_accum = select_time(surface_accum)
pressure = select_time(xr.open_dataset(PRESSURE_PATH, engine="netcdf4")).sel(pressure_level=850)

surface_instant, surface_accum, pressure

(<xarray.Dataset> Size: 189kB
 Dimensions:     (valid_time: 12, latitude: 17, longitude: 21)
 Coordinates:
   * valid_time  (valid_time) datetime64[ns] 96B 2017-03-14T04:00:00 ... 2017-...
     expver      (valid_time) <U4 192B ...
   * latitude    (latitude) float64 136B 41.0 40.75 40.5 ... 37.5 37.25 37.0
   * longitude   (longitude) float64 168B -97.0 -96.75 -96.5 ... -92.25 -92.0
     number      int64 8B ...
 Data variables:
     cl          (valid_time, latitude, longitude) float32 17kB ...
     sd          (valid_time, latitude, longitude) float32 17kB ...
     z           (valid_time, latitude, longitude) float32 17kB ...
     lsm         (valid_time, latitude, longitude) float32 17kB ...
     t2m         (valid_time, latitude, longitude) float32 17kB ...
     tcc         (valid_time, latitude, longitude) float32 17kB ...
     blh         (valid_time, latitude, longitude) float32 17kB ...
     u10         (valid_time, latitude, longitude) float32 17kB ...
     v10         (vali

In [4]:
def ordered3(da):
    drop_dims = [dim for dim in da.dims if dim not in {"valid_time", "latitude", "longitude"}]
    if drop_dims:
        da = da.isel({dim: 0 for dim in drop_dims})
    return da.transpose("valid_time", "latitude", "longitude")


def point_indices(da):
    da = ordered3(da)
    lat_index = int(da.latitude.to_index().get_indexer([POINT_LAT], method="nearest")[0])
    lon_index = int(da.longitude.to_index().get_indexer([POINT_LON], method="nearest")[0])
    return lat_index, lon_index


def neighborhood(da, radius=1):
    da = ordered3(da)
    lat_index, lon_index = point_indices(da)
    lat_slice = slice(max(lat_index - radius, 0), min(lat_index + radius + 1, da.sizes["latitude"]))
    lon_slice = slice(max(lon_index - radius, 0), min(lon_index + radius + 1, da.sizes["longitude"]))
    return da.isel(latitude=lat_slice, longitude=lon_slice).load()


def point_values(da):
    return np.asarray(neighborhood(da, radius=0).squeeze().values, dtype=float).reshape(-1)


def central_gradient(temp_neighborhood):
    lat_values = temp_neighborhood.latitude.values
    lon_values = temp_neighborhood.longitude.values
    center_lat = float(lat_values[len(lat_values) // 2])
    dlat = abs(float(lat_values[0] - lat_values[-1])) / (len(lat_values) - 1)
    dlon = abs(float(lon_values[-1] - lon_values[0])) / (len(lon_values) - 1)
    dy = R_EARTH * np.deg2rad(dlat)
    dx = R_EARTH * np.cos(np.deg2rad(center_lat)) * np.deg2rad(dlon)

    values = np.asarray(temp_neighborhood.values, dtype=float)
    center = len(lat_values) // 2
    west = len(lon_values) // 2 - 1
    east = len(lon_values) // 2 + 1
    north = center - 1
    south = center + 1

    dtdx = (values[:, center, east] - values[:, center, west]) / (2 * dx)
    dtdy = (values[:, north, center] - values[:, south, center]) / (2 * dy)
    return dtdx, dtdy


def center_values(da):
    values = np.asarray(da.values, dtype=float)
    center_y = values.shape[1] // 2
    center_x = values.shape[2] // 2
    return values[:, center_y, center_x]


def advection(temp, u, v):
    temp_neighborhood = neighborhood(temp, radius=1)
    u_neighborhood = neighborhood(u, radius=1)
    v_neighborhood = neighborhood(v, radius=1)
    dtdx, dtdy = central_gradient(temp_neighborhood)
    tendency_k_per_hr = -(center_values(u_neighborhood) * dtdx + center_values(v_neighborhood) * dtdy) * 3600.0
    return tendency_k_per_hr

In [5]:
time_index = pd.to_datetime(surface_instant.valid_time.values)
hour_labels = [timestamp.strftime("%Y-%m-%d %H:%M") for timestamp in time_index]

era5_fields = pd.DataFrame(
    {
        "valid_time_utc": hour_labels,
        "t2m_C": point_values(surface_instant["t2m"]) - 273.15,
        "t850_C": point_values(pressure["t"]) - 273.15,
        "u10_mps": point_values(surface_instant["u10"]),
        "v10_mps": point_values(surface_instant["v10"]),
        "u850_mps": point_values(pressure["u"]),
        "v850_mps": point_values(pressure["v"]),
        "omega850_Pa_s": point_values(pressure["w"]),
        "q850_kg_kg": point_values(pressure["q"]),
        "cloud_cover_pct": point_values(surface_instant["tcc"]) * 100.0,
        "down_sw_W_m2": point_values(surface_accum["ssrd"]) / 3600.0,
        "down_lw_W_m2": point_values(surface_accum["strd"]) / 3600.0,
        "apcp_mm": point_values(surface_accum["tp"]) * 1000.0,
        "pwat_kg_m2": point_values(surface_instant["tcwv"]),
        "pbl_height_m": point_values(surface_instant["blh"]),
        "snow_depth_m_we": point_values(surface_instant["sd"]),
        "soilw_top_m3_m3": point_values(surface_instant["swvl1"]),
        "terrain_m": point_values(surface_instant["z"]) / G,
        "lake_cover_frac": point_values(surface_instant["cl"]),
        "land_sea_mask_frac": point_values(surface_instant["lsm"]),
    },
    index=pd.Index(ANALYSIS_HOURS, name="hour_utc"),
)

era5_fields["wind10_mps"] = np.hypot(era5_fields["u10_mps"], era5_fields["v10_mps"])
era5_fields["wind850_mps"] = np.hypot(era5_fields["u850_mps"], era5_fields["v850_mps"])
era5_fields["net_down_rad_W_m2"] = era5_fields["down_sw_W_m2"] + era5_fields["down_lw_W_m2"]
era5_fields["adv_2m_K_per_hr"] = advection(surface_instant["t2m"], surface_instant["u10"], surface_instant["v10"])
era5_fields["adv_850_K_per_hr"] = advection(pressure["t"], pressure["u"], pressure["v"])

era5_fields_display = era5_fields.round(3)
era5_fields_display

,valid_time_utc,t2m_C,t850_C,u10_mps,v10_mps,u850_mps,v850_mps,omega850_Pa_s,q850_kg_kg,cloud_cover_pct,down_sw_W_m2,down_lw_W_m2,apcp_mm,pwat_kg_m2,pbl_height_m,snow_depth_m_we,soilw_top_m3_m3,terrain_m,lake_cover_frac,land_sea_mask_frac,wind10_mps,wind850_mps,net_down_rad_W_m2,adv_2m_K_per_hr,adv_850_K_per_hr
hour_utc,,,,,,,,,,,,,,,,,,,,,,,,,
4,2017-03-14 04:00,-1.662,-11.841,0.382,-3.945,2.106,-4.795,0.047,0.002,100.000,0.000,282.523,0.000,8.560,773.483,0.0,0.315,286.372,0.013,0.987,3.963,5.237,282.523,-0.136,-0.232
5,2017-03-14 05:00,-2.057,-12.056,0.296,-3.970,1.778,-5.037,0.061,0.002,100.000,0.000,285.696,0.000,8.723,799.519,0.0,0.315,286.372,0.013,0.987,3.981,5.341,285.696,-0.129,-0.362
6,2017-03-14 06:00,-2.313,-12.095,0.412,-3.829,1.943,-5.173,0.130,0.001,100.000,0.000,279.430,0.000,8.635,814.322,0.0,0.315,286.372,0.013,0.987,3.851,5.526,279.430,-0.144,-0.338
7,2017-03-14 07:00,-2.500,-11.802,0.201,-3.740,1.941,-5.518,0.120,0.001,100.000,0.000,285.145,0.006,8.367,844.352,0.0,0.315,286.372,0.013,0.987,3.745,5.850,285.145,-0.124,-0.488
8,2017-03-14 08:00,-2.629,-11.501,-0.109,-3.703,1.814,-5.995,0.152,0.001,100.000,0.000,271.990,0.005,8.216,873.319,0.0,0.314,286.372,0.013,0.987,3.704,6.263,271.990,-0.097,-0.574
9,2017-03-14 09:00,-2.913,-10.831,-0.579,-3.668,1.733,-6.635,0.230,0.001,100.000,0.000,269.411,0.006,8.264,961.221,0.0,0.314,286.372,0.013,0.987,3.714,6.857,269.411,-0.071,-0.620
10,2017-03-14 10:00,-3.451,-9.867,-1.047,-3.001,1.756,-6.710,0.317,0.001,98.676,0.000,254.491,0.001,9.143,728.966,0.0,0.316,286.372,0.013,0.987,3.178,6.936,254.491,-0.038,-0.440
11,2017-03-14 11:00,-3.624,-8.887,-1.274,-2.901,1.926,-7.718,0.156,0.001,99.451,0.000,245.565,0.000,9.678,688.195,0.0,0.316,286.372,0.013,0.987,3.169,7.955,245.565,-0.049,-0.482
12,2017-03-14 12:00,-3.640,-8.977,-1.542,-2.674,0.418,-7.925,-0.001,0.001,100.000,0.000,259.015,0.003,9.996,684.633,0.0,0.316,286.372,0.013,0.987,3.087,7.936,259.015,-0.065,-0.586


In [6]:
hourly = era5_fields.copy()
hourly["delta_t2m_C"] = hourly["t2m_C"].diff()
hourly["delta_t850_C"] = hourly["t850_C"].diff()
hourly["delta_cloud_pct"] = hourly["cloud_cover_pct"].diff()
hourly["delta_net_rad_W_m2"] = hourly["net_down_rad_W_m2"].diff()
hourly["delta_pwat_kg_m2"] = hourly["pwat_kg_m2"].diff()
hourly["delta_q850_g_kg"] = hourly["q850_kg_kg"].diff() * 1000.0
hourly["delta_pbl_m"] = hourly["pbl_height_m"].diff()
hourly["delta_soilw_top_m3_m3"] = hourly["soilw_top_m3_m3"].diff()
hourly["delta_snow_depth_m_we"] = hourly["snow_depth_m_we"].diff()

hourly["horizontal_adv_2m_C"] = hourly["adv_2m_K_per_hr"]
hourly["horizontal_adv_850_C"] = hourly["adv_850_K_per_hr"]
hourly["nonadv_residual_C"] = hourly["delta_t2m_C"] - hourly["horizontal_adv_2m_C"]

hourly_display = hourly.round(3)
hourly_display

,valid_time_utc,t2m_C,t850_C,u10_mps,v10_mps,u850_mps,v850_mps,omega850_Pa_s,q850_kg_kg,cloud_cover_pct,down_sw_W_m2,down_lw_W_m2,apcp_mm,pwat_kg_m2,pbl_height_m,snow_depth_m_we,soilw_top_m3_m3,terrain_m,lake_cover_frac,land_sea_mask_frac,wind10_mps,wind850_mps,net_down_rad_W_m2,adv_2m_K_per_hr,adv_850_K_per_hr,delta_t2m_C,delta_t850_C,delta_cloud_pct,delta_net_rad_W_m2,delta_pwat_kg_m2,delta_q850_g_kg,delta_pbl_m,delta_soilw_top_m3_m3,delta_snow_depth_m_we,horizontal_adv_2m_C,horizontal_adv_850_C,nonadv_residual_C
hour_utc,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,2017-03-14 04:00,-1.662,-11.841,0.382,-3.945,2.106,-4.795,0.047,0.002,100.000,0.000,282.523,0.000,8.560,773.483,0.0,0.315,286.372,0.013,0.987,3.963,5.237,282.523,-0.136,-0.232,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.136,-0.232,NaN
5,2017-03-14 05:00,-2.057,-12.056,0.296,-3.970,1.778,-5.037,0.061,0.002,100.000,0.000,285.696,0.000,8.723,799.519,0.0,0.315,286.372,0.013,0.987,3.981,5.341,285.696,-0.129,-0.362,-0.395,-0.216,0.000,3.173,0.163,-0.073,26.035,-0.000,0.0,-0.129,-0.362,-0.267
6,2017-03-14 06:00,-2.313,-12.095,0.412,-3.829,1.943,-5.173,0.130,0.001,100.000,0.000,279.430,0.000,8.635,814.322,0.0,0.315,286.372,0.013,0.987,3.851,5.526,279.430,-0.144,-0.338,-0.256,-0.038,0.000,-6.265,-0.088,-0.039,14.803,-0.000,0.0,-0.144,-0.338,-0.112
7,2017-03-14 07:00,-2.500,-11.802,0.201,-3.740,1.941,-5.518,0.120,0.001,100.000,0.000,285.145,0.006,8.367,844.352,0.0,0.315,286.372,0.013,0.987,3.745,5.850,285.145,-0.124,-0.488,-0.188,0.293,0.000,5.714,-0.268,-0.027,30.030,-0.000,0.0,-0.124,-0.488,-0.064
8,2017-03-14 08:00,-2.629,-11.501,-0.109,-3.703,1.814,-5.995,0.152,0.001,100.000,0.000,271.990,0.005,8.216,873.319,0.0,0.314,286.372,0.013,0.987,3.704,6.263,271.990,-0.097,-0.574,-0.129,0.301,0.000,-13.155,-0.151,-0.051,28.966,-0.000,0.0,-0.097,-0.574,-0.032
9,2017-03-14 09:00,-2.913,-10.831,-0.579,-3.668,1.733,-6.635,0.230,0.001,100.000,0.000,269.411,0.006,8.264,961.221,0.0,0.314,286.372,0.013,0.987,3.714,6.857,269.411,-0.071,-0.620,-0.283,0.670,0.000,-2.579,0.048,-0.065,87.903,-0.000,0.0,-0.071,-0.620,-0.213
10,2017-03-14 10:00,-3.451,-9.867,-1.047,-3.001,1.756,-6.710,0.317,0.001,98.676,0.000,254.491,0.001,9.143,728.966,0.0,0.316,286.372,0.013,0.987,3.178,6.936,254.491,-0.038,-0.440,-0.538,0.964,-1.324,-14.919,0.879,0.002,-232.255,0.002,0.0,-0.038,-0.440,-0.500
11,2017-03-14 11:00,-3.624,-8.887,-1.274,-2.901,1.926,-7.718,0.156,0.001,99.451,0.000,245.565,0.000,9.678,688.195,0.0,0.316,286.372,0.013,0.987,3.169,7.955,245.565,-0.049,-0.482,-0.173,0.980,0.775,-8.926,0.535,-0.024,-40.771,-0.000,0.0,-0.049,-0.482,-0.124
12,2017-03-14 12:00,-3.640,-8.977,-1.542,-2.674,0.418,-7.925,-0.001,0.001,100.000,0.000,259.015,0.003,9.996,684.633,0.0,0.316,286.372,0.013,0.987,3.087,7.936,259.015,-0.065,-0.586,-0.015,-0.090,0.549,13.450,0.318,0.017,-3.562,-0.000,0.0,-0.065,-0.586,0.049


In [7]:
def signed_percent(component, total):
    if pd.isna(total) or abs(total) < 1e-6:
        return np.nan
    return 100.0 * component / total


RD = 287.05
CP = 1004.0
P850_PA = 85000.0
AIR_DENSITY = 1.2
RADIATION_TO_AIR_FRACTION = 0.12


def estimate_non_advection_components(row):
    # These are deliberately independent, conservative estimates. They do not
    # use the observed residual, so the unexplained term can expose non-closure.
    pbl_depth = max(float(row["pbl_height_m"]), 50.0)
    pbl_coupling = np.clip(pbl_depth / 1500.0, 0.15, 0.75)
    air_mass_850 = pbl_coupling * row["delta_t850_C"]

    t850_k = row["t850_C"] + 273.15
    adiabatic_850_k_per_hr = row["omega850_Pa_s"] * RD * t850_k / (CP * P850_PA) * 3600.0
    vertical_motion = 0.25 * adiabatic_850_k_per_hr

    cloud_radiation = (
        row["delta_net_rad_W_m2"]
        * 3600.0
        / (AIR_DENSITY * CP * pbl_depth)
        * RADIATION_TO_AIR_FRACTION
    )

    moisture_precip = (
        0.03 * row["delta_pwat_kg_m2"]
        + 0.08 * row["delta_q850_g_kg"]
        - 0.08 * row["apcp_mm"]
    )

    pbl_change_factor = np.clip(row["delta_pbl_m"] / 500.0, -0.6, 0.6)
    mixing = 0.10 * pbl_change_factor * (row["t850_C"] - row["t2m_C"])

    surface = (
        -3.0 * row["delta_soilw_top_m3_m3"]
        - 20.0 * row["delta_snow_depth_m_we"]
    )

    return {
        "air_mass_850": air_mass_850,
        "vertical_motion": vertical_motion,
        "cloud_radiation": cloud_radiation,
        "moisture_precip": moisture_precip,
        "mixing": mixing,
        "surface": surface,
    }


def classify(row):
    surface_changed = abs(row["delta_t2m_C"]) >= 0.2
    upper_changed = abs(row["delta_t850_C"]) >= 0.2
    same_sign = np.sign(row["delta_t2m_C"]) == np.sign(row["delta_t850_C"])
    adv_same_sign = np.sign(row["horizontal_adv_2m_C"]) == np.sign(row["delta_t2m_C"])
    strong_adv = abs(row["horizontal_adv_2m_C"]) >= 0.2 and adv_same_sign
    strong_omega = abs(row["omega850_Pa_s"]) >= 0.25

    if surface_changed and upper_changed and same_sign and strong_adv:
        return "2m/850 changed together; advection/air-mass signal"
    if surface_changed and upper_changed and same_sign:
        return "2m/850 changed together; synoptic air-mass signal"
    if surface_changed and not upper_changed:
        return "surface/radiation/mixing/local signal"
    if upper_changed and not surface_changed:
        return "upper-level change muted near surface"
    if strong_omega:
        return "vertical-motion/cloud-process signal"
    return "weak or mixed signal"


def contribution_cell(value, pct):
    if pd.isna(value):
        return ""
    if pd.isna(pct):
        return f"{value:+.2f}"
    return f"{value:+.2f} ({pct:+.0f}%)"


rows = []
for hour, row in hourly.iterrows():
    if pd.isna(row["delta_t2m_C"]):
        continue

    estimated = estimate_non_advection_components(row)

    components = {
        "horizontal_advection": row["horizontal_adv_2m_C"],
        "air_mass_850": estimated["air_mass_850"],
        "vertical_motion": estimated["vertical_motion"],
        "cloud_radiation": estimated["cloud_radiation"],
        "moisture_precip": estimated["moisture_precip"],
        "mixing": estimated["mixing"],
        "surface": estimated["surface"],
    }
    explained = sum(components.values())
    unexplained = row["delta_t2m_C"] - explained

    rows.append(
        {
            "ending_hour_utc": int(hour),
            "ending_valid_utc": row["valid_time_utc"],
            "2m_temp_change_C": f"{row['delta_t2m_C']:+.2f}",
            "horizontal_advection": contribution_cell(
                components["horizontal_advection"],
                signed_percent(components["horizontal_advection"], row["delta_t2m_C"]),
            ),
            "air_mass_850": contribution_cell(
                components["air_mass_850"],
                signed_percent(components["air_mass_850"], row["delta_t2m_C"]),
            ),
            "vertical_motion": contribution_cell(
                components["vertical_motion"],
                signed_percent(components["vertical_motion"], row["delta_t2m_C"]),
            ),
            "cloud_radiation": contribution_cell(
                components["cloud_radiation"],
                signed_percent(components["cloud_radiation"], row["delta_t2m_C"]),
            ),
            "moisture_precip": contribution_cell(
                components["moisture_precip"],
                signed_percent(components["moisture_precip"], row["delta_t2m_C"]),
            ),
            "mixing": contribution_cell(
                components["mixing"],
                signed_percent(components["mixing"], row["delta_t2m_C"]),
            ),
            "surface": contribution_cell(
                components["surface"],
                signed_percent(components["surface"], row["delta_t2m_C"]),
            ),
            "unexplained": contribution_cell(
                unexplained,
                signed_percent(unexplained, row["delta_t2m_C"]),
            ),
            "850_temp_change_C": f"{row['delta_t850_C']:+.2f}",
            "850_advection_C": f"{row['horizontal_adv_850_C']:+.2f}",
            "omega850_Pa_s": f"{row['omega850_Pa_s']:+.3f}",
            "cloud_cover_pct": f"{row['cloud_cover_pct']:.1f}",
            "net_down_rad_W_m2": f"{row['net_down_rad_W_m2']:.1f}",
            "precip_mm": f"{row['apcp_mm']:.2f}",
            "pwat_kg_m2": f"{row['pwat_kg_m2']:.2f}",
            "wind10_mps": f"{row['wind10_mps']:.1f}",
            "pbl_height_m": f"{row['pbl_height_m']:.0f}",
            "snow_depth_m_we": f"{row['snow_depth_m_we']:.3f}",
            "soilw_top_m3_m3": f"{row['soilw_top_m3_m3']:.3f}",
            "terrain_m": f"{row['terrain_m']:.0f}",
            "lake_cover_frac": f"{row['lake_cover_frac']:.2f}",
            "interpretation": classify(row),
        }
    )

attribution = pd.DataFrame(rows)
attribution

,ending_hour_utc,ending_valid_utc,2m_temp_change_C,horizontal_advection,air_mass_850,vertical_motion,cloud_radiation,moisture_precip,mixing,surface,unexplained,850_temp_change_C,850_advection_C,omega850_Pa_s,cloud_cover_pct,net_down_rad_W_m2,precip_mm,pwat_kg_m2,wind10_mps,pbl_height_m,snow_depth_m_we,soilw_top_m3_m3,terrain_m,lake_cover_frac,interpretation
0,5,2017-03-14 05:00,-0.40,-0.13 (+33%),-0.11 (+29%),+0.05 (-12%),+0.00 (-0%),-0.00 (+0%),-0.05 (+13%),+0.00 (-0%),-0.15 (+38%),-0.22,-0.36,+0.061,100.0,285.7,0.00,8.72,4.0,800,0.000,0.315,286,0.01,2m/850 changed together; synoptic air-mass signal
1,6,2017-03-14 06:00,-0.26,-0.14 (+56%),-0.02 (+8%),+0.10 (-40%),-0.00 (+1%),-0.01 (+2%),-0.03 (+11%),+0.00 (-0%),-0.16 (+61%),-0.04,-0.34,+0.130,100.0,279.4,0.00,8.64,3.9,814,0.000,0.315,286,0.01,surface/radiation/mixing/local signal
2,7,2017-03-14 07:00,-0.19,-0.12 (+66%),+0.16 (-88%),+0.09 (-51%),+0.00 (-1%),-0.01 (+6%),-0.06 (+30%),+0.00 (-0%),-0.26 (+139%),+0.29,-0.49,+0.120,100.0,285.1,0.01,8.37,3.7,844,0.000,0.315,286,0.01,upper-level change muted near surface
3,8,2017-03-14 08:00,-0.13,-0.10 (+75%),+0.17 (-135%),+0.12 (-93%),-0.01 (+4%),-0.01 (+7%),-0.05 (+40%),+0.00 (-1%),-0.26 (+203%),+0.30,-0.57,+0.152,100.0,272.0,0.01,8.22,3.7,873,0.000,0.314,286,0.01,upper-level change muted near surface
4,9,2017-03-14 09:00,-0.28,-0.07 (+25%),+0.43 (-152%),+0.18 (-65%),-0.00 (+0%),-0.00 (+1%),-0.14 (+49%),+0.00 (-0%),-0.68 (+240%),+0.67,-0.62,+0.230,100.0,269.4,0.01,8.26,3.7,961,0.000,0.314,286,0.01,weak or mixed signal
5,10,2017-03-14 10:00,-0.54,-0.04 (+7%),+0.47 (-87%),+0.25 (-47%),-0.01 (+1%),+0.03 (-5%),+0.30 (-55%),-0.01 (+1%),-1.53 (+285%),+0.96,-0.44,+0.317,98.7,254.5,0.00,9.14,3.2,729,0.000,0.316,286,0.01,vertical-motion/cloud-process signal
6,11,2017-03-14 11:00,-0.17,-0.05 (+29%),+0.45 (-260%),+0.12 (-72%),-0.00 (+3%),+0.01 (-8%),+0.04 (-25%),+0.00 (-0%),-0.75 (+434%),+0.98,-0.48,+0.156,99.5,245.6,0.00,9.68,3.2,688,0.000,0.316,286,0.01,upper-level change muted near surface
7,12,2017-03-14 12:00,-0.02,-0.06 (+420%),-0.04 (+267%),-0.00 (+6%),+0.01 (-46%),+0.01 (-69%),+0.00 (-25%),+0.00 (-4%),+0.07 (-450%),-0.09,-0.59,-0.001,100.0,259.0,0.00,10.00,3.1,685,0.000,0.316,286,0.01,weak or mixed signal
8,13,2017-03-14 13:00,-0.04,-0.06 (+151%),-0.34 (+822%),-0.11 (+272%),+0.01 (-19%),-0.00 (+0%),-0.04 (+107%),+0.00 (-1%),+0.51 (-1232%),-0.71,-0.45,-0.142,100.0,275.1,0.02,10.06,2.9,722,0.000,0.315,286,0.01,upper-level change muted near surface
9,14,2017-03-14 14:00,+0.07,-0.05 (-74%),-0.67 (-966%),-0.16 (-231%),+0.03 (+48%),-0.00 (-0%),+0.02 (+24%),+0.00 (+1%),+0.90 (+1298%),-1.41,-0.43,-0.201,100.0,341.0,0.07,10.22,2.3,711,0.000,0.315,286,0.01,upper-level change muted near surface


## Reading the Table

Contribution columns are shown as `temperature contribution (percent of observed 2 m temperature change)`. A negative percent means that component opposed the observed 2 m temperature change.

The directly computed column is `horizontal_advection`. The other contribution columns are independent, deliberately simple estimates from the ERA5 diagnostics. `unexplained` is the remaining difference after summing the displayed components, so it is a closure check rather than a value forced to zero.